# Chapter 17 – Worked examples

This notebook runs the examples of [Chapter 17](https://katra.ballardini.com.ar/power-grid-topology-optimization-course/0.2.00.2.0/part-3-power-systems/ch17-sensitivity-factors-1/) live. The code lives in the modules next to it
(`lodf.py`, `n1_screening.py`, `triangle.py`); the cells only call them, so a change in a module is a change here
too. The Octave examples (`matpower_lodf.m`, `ptdf_slack_case30.m`) run in Octave, not here.

In [ ]:
# JupyterLite only: install the libraries this notebook and its chapter modules use.
%pip install -q colorama==0.4.6 deepdiff==8.6.2 geojson==3.3.0 matplotlib networkx numpy packaging pandas pandera==0.34.1 scipy tqdm typing_extensions tzdata
%pip install -q --no-deps pandapower==3.5.6

In [ ]:
import lodf
import matplotlib.pyplot as plt
import n1_screening
import numpy as np
import triangle

## PTDF and N-1 flows of a four-bus grid

One generator at bus 0, one load at bus 3, five branches. `ptdf_matrix` solves one linear system for all the
non-slack columns; `n1_flows` applies $F^{(k)} = F^{(0)} + \text{LODF}_{:,k}F^{(0)}_k$ for every outage.

In [ ]:
frm, to = np.array([0, 0, 1, 1, 2]), np.array([1, 2, 2, 3, 3])
x = np.array([0.1, 0.2, 0.1, 0.2, 0.1])
p = np.array([100.0, 0.0, 0.0, -100.0])
h = lodf.ptdf_matrix(frm, to, x, 4)
print('PTDF (slack column 0 is zero):')
print(np.round(h, 3))
print('base flows      :', np.round(h @ p, 1))
print('N-1 flows (row = outaged branch):')
print(np.round(lodf.n1_flows(h, frm, to, p), 1))

## LODF by compensation on the triangle

Equal reactances, slack at bus 3. Every $h_{kk}$ is $2/3$ (the other two branches in series have twice the
reactance), so $\text{LODF}_{\ell k} = h_{\ell k}/(1 - h_{kk}) = \pm 1$: after an outage the whole flow of the
lost branch goes through the two remaining ones in series.

In [ ]:
print(triangle.ptdf())
print(triangle.lodf())

## Slack dependence

$H_w = H_s(I - w\mathbf 1^\top)$ changes every column by the same vector $-H_s w$, so the flows of a balanced
injection and the LODF do not change; the flows after a generator outage do, because the lost power is picked up at
the slack (single) or by every bus in proportion to $w$ (distributed). IEEE 14 from pandapower.

In [ ]:
frm14, to14, x14, p14, slack14 = n1_screening.case_data('case14')
h14 = lodf.ptdf_matrix(frm14, to14, x14, 14, slack14)
w = np.full(14, 1 / 14)
hw = n1_screening.distributed_slack_ptdf(h14, w)
print(
    f'max |PTDF change| {np.abs(hw - h14).max():.3f} | max |flow change| {np.abs(hw @ p14 - h14 @ p14).max():.1e} MW'
)
single = n1_screening.generator_outage(h14, p14, 1, 40.0)
shared = n1_screening.generator_outage(h14, p14, 1, 40.0, w)
plt.figure(figsize=(8, 3))
plt.bar(range(20), single - shared)
plt.xlabel('branch')
plt.ylabel('MW')
plt.title('Loss of 40 MW at bus 2: single-slack minus distributed-slack post-outage flow')
plt.show()

## Bridges

A bridge has $h_{kk} = 1$ and no LODF column. IEEE 14 has one (branch 7–8, the transformer that feeds bus 8 alone);
`lodf_matrix` returns NaN there and `brute_force_n1` cannot solve the islanded grid either.

In [ ]:
lodf14 = n1_screening.lodf_matrix(h14, frm14, to14)
print(
    'bridges:', n1_screening.bridges(frm14, to14, 14), '| NaN columns:', np.flatnonzero(np.isnan(lodf14).all(axis=0))
)
print('1 - h_kk on the bridge:', 1 - np.diag(n1_screening.transfer_factors(h14, frm14, to14))[18])